# Jamboree Graduate Admission Analysis
### A documented regression workflow for admission-chance estimation

**Audience:** Jamboree product, counseling, and analytics stakeholders  
**Objective:** Explore applicant-profile data, identify associations with estimated admission chance, and build a reproducible baseline prediction model.

This notebook analyzes the provided historical admission-chance dataset. Its target is an estimated chance score, not a verified admission outcome. Results describe patterns in this dataset and should not be interpreted as causal effects or admissions guarantees.

> **Reproducibility note:** This notebook has been rebuilt to address issues found in the earlier version. Run every cell from top to bottom to generate fresh tables, diagnostics, and metrics. Previous saved notebook outputs were internally inconsistent, so they have been cleared rather than carried forward as if verified.

## 1. Business context and questions

Jamboree offers students guidance as they prepare applications to graduate programs. An admission-chance estimate can help learners understand how their current profile compares with patterns in the available data.

This analysis addresses four questions:

1. Which observed applicant features are associated with Chance of Admit?
2. How strongly are the predictor variables related to one another?
3. How well do linear models estimate chance on applicants held out from training?
4. What data, validation, and product safeguards would be needed before real-world use?

### Decision boundary

The dataset contains a continuous chance estimate rather than confirmed admission decisions. The model therefore estimates the supplied chance score; it does not predict a university's final decision or establish that a student will be admitted.

## 2. Dataset and feature definitions

The source file is Jamboree.csv in the repository root. It contains 500 rows, a unique serial identifier, seven applicant-profile predictors, and the target.

| Field | Description |
| --- | --- |
| Serial No. | Row identifier; excluded from modeling |
| GRE Score | GRE score, out of 340 |
| TOEFL Score | TOEFL score, out of 120 |
| University Rating | University rating, from 1 to 5 |
| SOP | Statement of Purpose strength, from 1 to 5 |
| LOR | Letter of Recommendation strength, from 1 to 5 |
| CGPA | Undergraduate GPA, out of 10 |
| Research | Research experience indicator, 0 or 1 |
| Chance of Admit | Estimated admission chance, from 0 to 1; target |

University Rating, SOP, and LOR are ordinal ratings. Research is binary. This notebook keeps them numeric to match the assignment's linear-regression framing; this choice assumes the numeric steps are sufficiently meaningful for a baseline linear model.

## 3. Environment and reproducibility

The analysis uses common Python data-science packages. Install them in the active environment if they are not already available. A fixed random seed makes the holdout split repeatable.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset
from statsmodels.graphics.gofplots import qqplot
from scipy.stats import shapiro, jarque_bera

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 2
TEST_SIZE = 0.20
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)
warnings.filterwarnings("ignore", category=FutureWarning)

## 4. Load the data

Load the repository copy of the CSV. Column names are stripped of accidental leading or trailing spaces so later code can use consistent names. The fallback supports the lowercase filename used by the original Colab notebook.

In [ ]:
candidate_paths = [Path("Jamboree.csv"), Path("jamboree.csv")]
data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Place Jamboree.csv (or jamboree.csv) in the same folder as this notebook."
    )

df = pd.read_csv(data_path)
df.columns = df.columns.str.strip()

print("Source:", data_path.resolve())
print("Rows and columns:", df.shape)
display(df.head())

## 5. Data quality and structure

Check the schema, missingness, duplicate rows, duplicate identifiers, and value ranges before modeling. A clean null count does not replace checking duplicates or validating that values fall within their documented ranges.

In [ ]:
print("Data types and non-null counts")
display(df.info())

print("Missing values by column")
display(df.isna().sum().rename("missing_count").to_frame())

print("Exact duplicate rows:", int(df.duplicated().sum()))
if "Serial No." in df.columns:
    print("Duplicate serial identifiers:", int(df["Serial No."].duplicated().sum()))

print("Descriptive statistics")
display(df.describe().T)

### Data-quality interpretation

The original notebook reported 500 complete rows and no null values. This version independently checks those properties and also checks duplicates. It does not silently impute or remove records: any data-quality action should be based on the output of these checks and documented before modeling.

In [ ]:
required_columns = {
    "Serial No.", "GRE Score", "TOEFL Score", "University Rating",
    "SOP", "LOR", "CGPA", "Research", "Chance of Admit"
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Expected columns are missing: {sorted(missing_columns)}")

assert df["Serial No."].is_unique, "Serial No. is expected to be a unique row identifier."
assert df["Chance of Admit"].between(0, 1).all(), "Target values should lie between 0 and 1."
assert df["Research"].isin([0, 1]).all(), "Research should be coded as 0 or 1."

range_checks = pd.DataFrame({
    "minimum": df.drop(columns="Serial No.").min(),
    "maximum": df.drop(columns="Serial No.").max()
})
display(range_checks)

## 6. Exploratory data analysis

Review feature distributions and the target before fitting a model. The discrete ratings and research indicator are shown as counts; score-like variables and the target are shown as distributions. These plots help identify skew, limited ranges, unusual values, and data-entry concerns.

In [ ]:
continuous_columns = [
    "GRE Score", "TOEFL Score", "CGPA", "Chance of Admit"
]
ordinal_columns = ["University Rating", "SOP", "LOR", "Research"]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for column, ax in zip(continuous_columns, axes.flat):
    sns.histplot(data=df, x=column, kde=True, ax=ax)
    ax.set_title(f"Distribution of {column}")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for column, ax in zip(ordinal_columns, axes.flat):
    sns.countplot(data=df, x=column, ax=ax)
    ax.set_title(f"Counts for {column}")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for column, ax in zip(continuous_columns + ordinal_columns[:3], axes.flat):
    sns.boxplot(data=df, x=column, ax=ax)
    ax.set_title(f"Range and potential outliers: {column}")
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()

### Outlier review and treatment decision

The earlier notebook's outlier function computed an IQR upper bound but only replaced values below the lower bound, and it modified both LOR and the target. That behavior was asymmetric and could distort the outcome variable.

This version flags potential IQR outliers for review but does not automatically cap or delete them. A flagged value is not necessarily an error. Investigate data provenance and assess influential observations before choosing a treatment; keep the target on its original 0–1 scale.

In [ ]:
def iqr_outlier_summary(data, columns):
    rows = []
    for column in columns:
        q1 = data[column].quantile(0.25)
        q3 = data[column].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        flagged = data[column].lt(lower) | data[column].gt(upper)
        rows.append({
            "feature": column,
            "lower_fence": lower,
            "upper_fence": upper,
            "flagged_rows": int(flagged.sum())
        })
    return pd.DataFrame(rows)

outlier_summary = iqr_outlier_summary(
    df, continuous_columns + ["University Rating", "SOP", "LOR"]
)
display(outlier_summary)

## 7. Relationships and correlation

A correlation matrix gives a compact view of linear relationships among numeric fields. A pair plot provides a visual complement. Correlation is descriptive; it does not establish causality and does not capture every nonlinear relationship.

In [ ]:
predictor_columns = [
    "GRE Score", "TOEFL Score", "University Rating",
    "SOP", "LOR", "CGPA", "Research"
]
target_column = "Chance of Admit"
analysis_columns = predictor_columns + [target_column]

corr = df[analysis_columns].corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=True)
plt.title("Correlation matrix: applicant features and admission chance")
plt.tight_layout()
plt.show()

display(corr[target_column].drop(target_column).sort_values(ascending=False)
        .rename("correlation_with_chance").to_frame())

In [ ]:
sns.pairplot(
    df[["GRE Score", "TOEFL Score", "CGPA", "SOP", "LOR", target_column]],
    corner=True,
    diag_kind="hist",
    plot_kws={"alpha": 0.55, "s": 20}
)
plt.suptitle("Pairwise relationships among selected features", y=1.02)
plt.show()

### EDA interpretation prompt

Use the displayed plots and correlation table to record which features appear most related to the target, which predictors move together, and whether any pattern looks nonlinear. The following regression analysis estimates conditional associations after accounting for the other included predictors; those results can differ from simple pairwise correlations.

## 8. Define the modeling data and split

Remove the unique serial identifier so the model cannot use row order as a predictive signal. Keep the target separate from the predictors. Split before fitting any transformations, and use a fixed random state for reproducibility.

In [ ]:
model_df = df.drop(columns=["Serial No."]).copy()
X = model_df[predictor_columns].copy()
y = model_df[target_column].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Predictors: {len(predictor_columns)}")

## 9. Fit preprocessing on training data only

OLS coefficients are easier to compare when the features use a common scale. Fit MinMaxScaler on the training predictors, then apply that same fitted scaler to the test predictors. This corrects the earlier notebook's test-set leakage and inconsistent feature mapping.

In [ ]:
minmax_scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(
    minmax_scaler.fit_transform(X_train),
    columns=predictor_columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    minmax_scaler.transform(X_test),
    columns=predictor_columns,
    index=X_test.index
)

display(X_train_scaled.describe().T[["min", "max", "mean", "std"]])

## 10. OLS baseline with Statsmodels

Fit an OLS model using all seven predictors. The model summary provides coefficient estimates, confidence intervals, p-values, and overall fit statistics. Because the predictors are scaled, each coefficient corresponds to a one-unit change on the scaled range, holding the other predictors constant.

In [ ]:
X_train_ols = sm.add_constant(X_train_scaled, has_constant="add")
X_test_ols = sm.add_constant(X_test_scaled, has_constant="add")

ols_full = sm.OLS(y_train, X_train_ols).fit()
print(ols_full.summary())

In [ ]:
coefficient_table = pd.DataFrame({
    "coefficient": ols_full.params,
    "std_error": ols_full.bse,
    "t_statistic": ols_full.tvalues,
    "p_value": ols_full.pvalues
})
confidence_intervals = ols_full.conf_int()
coefficient_table["ci_lower"] = confidence_intervals[0]
coefficient_table["ci_upper"] = confidence_intervals[1]
display(coefficient_table.sort_values("p_value"))

### How to read the coefficients

A positive coefficient means the model estimates a higher Chance of Admit as that feature increases, conditional on the other predictors. A small p-value is evidence against a zero coefficient under the model assumptions; it does not show causation or guarantee practical importance. When predictors are highly correlated, their individual coefficients can be unstable even when the model predicts reasonably well.

## 11. Multicollinearity and VIF review

Variance Inflation Factor (VIF) quantifies how strongly a predictor can be explained by the other predictors. A high value signals overlapping information and can make OLS coefficient estimates difficult to interpret.

The original workflow stopped before all VIF values were below 5 and selected predictors through a condition tied to adjusted R². Here the initial VIF table is reported transparently, then a clearly defined train-only greedy reduction removes the current highest-VIF feature until the threshold is met or only one predictor remains. Both the full and reduced OLS models are retained for comparison; the reduced model is not automatically assumed to be better.

In [ ]:
def calculate_vif(data):
    values = data.astype(float)
    result = pd.DataFrame({
        "feature": values.columns,
        "VIF": [
            variance_inflation_factor(values.to_numpy(), i)
            for i in range(values.shape[1])
        ]
    })
    return result.sort_values("VIF", ascending=False).reset_index(drop=True)

initial_vif = calculate_vif(X_train_scaled)
display(initial_vif)

In [ ]:
VIF_THRESHOLD = 5.0
vif_features = list(predictor_columns)
vif_steps = []

while len(vif_features) > 1:
    current_vif = calculate_vif(X_train_scaled[vif_features])
    largest_vif = float(current_vif.iloc[0]["VIF"])
    if largest_vif <= VIF_THRESHOLD:
        break
    feature_to_remove = str(current_vif.iloc[0]["feature"])
    vif_steps.append({
        "removed_feature": feature_to_remove,
        "VIF_at_removal": largest_vif
    })
    vif_features.remove(feature_to_remove)

final_vif = calculate_vif(X_train_scaled[vif_features])
print("VIF-based feature removal sequence")
display(pd.DataFrame(vif_steps))
print("VIFs after reduction")
display(final_vif)
print("Threshold reached:", bool(final_vif["VIF"].max() <= VIF_THRESHOLD))

### Reduced OLS model

The VIF reduction is computed from training predictors only. This model is a diagnostic comparison, not a claim that features with high VIF are unimportant. Compare fit and holdout performance before selecting it. The full OLS model remains the primary reference for interpreting all predictors.

In [ ]:
X_train_vif = sm.add_constant(
    X_train_scaled[vif_features], has_constant="add"
)
X_test_vif = sm.add_constant(
    X_test_scaled[vif_features], has_constant="add"
)
ols_vif = sm.OLS(y_train, X_train_vif).fit()
print(ols_vif.summary())

## 12. Regression assumption diagnostics

Diagnostics are calculated for the full OLS model's training residuals. These checks assess whether standard linear-model interpretation is reasonable; no single plot or test proves the assumptions hold.

- **Mean residual:** should be close to zero when an intercept is included.
- **Linearity:** inspect residuals versus fitted values for systematic curves or structure; the RESET test provides an additional screen for functional-form misspecification.
- **Homoscedasticity:** inspect residual spread and use the Breusch–Pagan test.
- **Normality:** use a histogram and Q–Q plot, alongside Shapiro–Wilk and Jarque–Bera tests. With 400 observations, formal tests can flag small departures, so consider the plots and model purpose as well.

In [ ]:
train_fitted = ols_full.predict(X_train_ols)
train_residuals = y_train - train_fitted

print("Mean training residual:", float(train_residuals.mean()))
print("Breusch-Pagan LM statistic and p-value:",
      het_breuschpagan(train_residuals, X_train_ols)[:2])
print("Ramsey RESET test:")
display(linear_reset(ols_full, power=2, use_f=True))
print("Shapiro-Wilk statistic and p-value:",
      shapiro(train_residuals))
print("Jarque-Bera statistic and p-value:",
      jarque_bera(train_residuals))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.scatterplot(x=train_fitted, y=train_residuals, alpha=0.65, ax=axes[0, 0])
axes[0, 0].axhline(0, color="red", linestyle="--")
axes[0, 0].set(title="Residuals versus fitted values",
               xlabel="Fitted Chance of Admit", ylabel="Residual")

sns.histplot(train_residuals, kde=True, ax=axes[0, 1])
axes[0, 1].set(title="Training residual distribution", xlabel="Residual")

qqplot(train_residuals, line="45", ax=axes[1, 0])
axes[1, 0].set_title("Normal Q–Q plot")

sns.scatterplot(x=train_fitted, y=np.sqrt(np.abs(train_residuals)),
                alpha=0.65, ax=axes[1, 1])
axes[1, 1].set(title="Scale-location diagnostic",
               xlabel="Fitted Chance of Admit",
               ylabel="Square root of absolute residual")
plt.tight_layout()
plt.show()

## 13. Evaluation metrics

Report MAE, RMSE, R², and adjusted R² for both training and held-out test predictions. MAE and RMSE are in Chance of Admit units; for example, an error of 0.05 is five percentage points on the 0–1 scale. RMSE is calculated as the square root of mean squared error. Adjusted R² is included with the number of predictors shown in the table.

Training metrics describe fit to data used to estimate the model. Test metrics provide a more useful, though still limited, estimate of performance on unseen records.

In [ ]:
def regression_metrics(y_true, y_pred, n_features):
    n = len(y_true)
    r2 = r2_score(y_true, y_pred)
    adjusted_r2 = (
        1 - (1 - r2) * (n - 1) / (n - n_features - 1)
        if n > n_features + 1 else np.nan
    )
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2,
        "Adjusted R2": adjusted_r2
    }

def evaluate_predictions(model_name, y_train_true, y_train_pred,
                         y_test_true, y_test_pred, n_features):
    rows = []
    for split, actual, predicted in [
        ("Train", y_train_true, y_train_pred),
        ("Test", y_test_true, y_test_pred)
    ]:
        rows.append({
            "Model": model_name,
            "Split": split,
            "Predictors": n_features,
            **regression_metrics(actual, predicted, n_features)
        })
    return rows

full_train_pred = ols_full.predict(X_train_ols)
full_test_pred = ols_full.predict(X_test_ols)
vif_train_pred = ols_vif.predict(X_train_vif)
vif_test_pred = ols_vif.predict(X_test_vif)

metric_rows = []
metric_rows += evaluate_predictions(
    "OLS (all features)", y_train, full_train_pred,
    y_test, full_test_pred, len(predictor_columns)
)
metric_rows += evaluate_predictions(
    "OLS (VIF reduced)", y_train, vif_train_pred,
    y_test, vif_test_pred, len(vif_features)
)

## 14. Regularized regression: Ridge and Lasso

Ridge and Lasso add a penalty to reduce coefficient instability when predictors overlap. Hyperparameters are selected using five-fold cross-validation within the training set. Each cross-validation fold fits its own scaler through the pipeline, and the untouched test set is used only for final evaluation.

In [ ]:
alpha_grid = {"model__alpha": np.logspace(-4, 3, 60)}

ridge_search = GridSearchCV(
    Pipeline([
        ("scale", StandardScaler()),
        ("model", Ridge())
    ]),
    param_grid=alpha_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)
ridge_search.fit(X_train, y_train)

lasso_search = GridSearchCV(
    Pipeline([
        ("scale", StandardScaler()),
        ("model", Lasso(max_iter=100_000))
    ]),
    param_grid=alpha_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)
lasso_search.fit(X_train, y_train)

print("Selected Ridge alpha:", ridge_search.best_params_["model__alpha"])
print("Selected Lasso alpha:", lasso_search.best_params_["model__alpha"])

In [ ]:
for model_name, search in [
    ("Ridge (CV tuned)", ridge_search),
    ("Lasso (CV tuned)", lasso_search)
]:
    metric_rows += evaluate_predictions(
        model_name,
        y_train, search.predict(X_train),
        y_test, search.predict(X_test),
        len(predictor_columns)
    )

metrics = pd.DataFrame(metric_rows)
display(metrics.sort_values(["Split", "RMSE"]))

### Model comparison and selection

Compare held-out test MAE and RMSE, while checking the train/test gap and coefficient stability. The lowest test error is not automatically the best product choice: interpretability, calibration, subgroup performance, and the consequences of incorrect advice also matter. Do not tune repeatedly against the test set; use cross-validation on training data for model selection.

In [ ]:
best_test = (
    metrics.loc[metrics["Split"].eq("Test")]
    .sort_values("RMSE")
    .iloc[0]
)
print("Lowest test RMSE in this comparison:",
      best_test["Model"], f"({best_test['RMSE']:.4f})")

## 15. Interpreting model effects

Review the OLS coefficient table with the correlation and VIF results together. A feature can show a positive pairwise correlation with chance but a weak conditional coefficient after other correlated variables enter the model. High VIF indicates that the model has limited information to separate the independent contribution of those features.

For regularized models, coefficients are estimated after standardization. The next cell displays their directions and relative magnitudes; do not compare them numerically with MinMax-scaled OLS coefficients without accounting for the different scaling.

In [ ]:
regularized_coefficients = pd.DataFrame({
    "feature": predictor_columns,
    "Ridge coefficient (standardized)": ridge_search.best_estimator_
        .named_steps["model"].coef_,
    "Lasso coefficient (standardized)": lasso_search.best_estimator_
        .named_steps["model"].coef_
})
display(regularized_coefficients.sort_values(
    "Ridge coefficient (standardized)", ascending=False
))

## 16. Stakeholder findings and recommended next steps

### What this analysis can support

- Describe which available profile variables are associated with the supplied admission-chance estimates.
- Provide an exploratory benchmark for a student-facing estimate, after validating the corrected holdout results.
- Identify data gaps and correlated inputs that need attention before deployment.

### What it cannot establish

- It cannot prove that changing GRE, TOEFL, CGPA, or application materials causes a specific admission result.
- It cannot guarantee admission to an Ivy League or any other university.
- It cannot estimate actual admission outcomes unless the training target is verified outcome data.

### Recommendations

1. Re-run the notebook from top to bottom and use the corrected train-only preprocessing and fresh metrics.
2. Collect confirmed admission decisions and program/university context, then validate against outcomes that match the intended product use.
3. Expand the data across institutions, programs, applicant cohorts, and years; document collection method and target construction.
4. Report prediction error and calibration, and inspect performance across relevant applicant groups before showing estimates to students.
5. Present the estimate with uncertainty and plain-language limitations; use it to support counseling rather than replace expert guidance or institutional decisions.
6. Monitor performance over time and define a process for updating, reviewing, or withdrawing the model when its quality changes.

### Final interpretation

This is an educational baseline for continuous regression on a small dataset. Stakeholder use should wait until the notebook has been run successfully, the corrected metrics are reviewed, the target's provenance is verified, and broader validation demonstrates that predictions are useful and appropriately communicated.

## 17. Reproducibility checklist

- [ ] Confirm the source CSV is the intended version and document its provenance.
- [ ] Run all cells in order from a clean kernel.
- [ ] Confirm data-quality checks and investigate any flagged rows.
- [ ] Review correlation and VIF outputs; document any feature reduction.
- [ ] Review residual plots and formal diagnostic test results.
- [ ] Compare train and test MAE, RMSE, R², and adjusted R².
- [ ] Record the final model choice and limitations before sharing results.